# Compare different spike type prediction algorithms

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
MAIN_PATH = 'E:/Data/BogiSejtek/Method_Comparison'
BASE_PATH = f'{MAIN_PATH}/Base'
CHAO_PATH = f'{MAIN_PATH}/Chao'
COMBINED_PATH = f'{MAIN_PATH}/Combined'
CELL_TYPE = 'RSI_L2-3'
CELL_NAME = 'B18_d3t1c1'

In [ ]:
base_df = pd.read_csv(f'{BASE_PATH}/{CELL_TYPE}/{CELL_NAME}.csv')
base_df = base_df[['timestamp', 'spike_type']]
base_df['spike_type'] = base_df['spike_type'].map({
    'AIS-AP': 'AIS',
    'UNKNOWN': 'UND',
    'RA-AP': 'RA'
})

In [ ]:
chao_df = pd.read_excel(f'{CHAO_PATH}/{CELL_TYPE}/{CELL_NAME}.xlsx')
chao_df = chao_df.iloc[:, -3:]
chao_df = chao_df.rename(columns={
    'spike_time_sec': 'timestamp',
    'label_Hard_4quadrant_086_017': 'hard_label',
    'label_Highcoverage_099_012': 'soft_label'
})

In [ ]:
combination = []
for _, row in base_df.iterrows():
    timestamp = row['timestamp']
    spike_type = row['spike_type']
    best_index = np.argmin(np.abs(chao_df['timestamp'] - timestamp))
    time_diff = abs(chao_df['timestamp'].iloc[best_index] - timestamp)
    if time_diff > 0.001:
        continue
    combination.append({
        'timestamp': timestamp,
        'spike_type': spike_type,
        'hard_label': chao_df['hard_label'].iloc[best_index],
        'soft_label': chao_df['soft_label'].iloc[best_index]
    })
comb_df = pd.DataFrame(combination)
Path(f'{COMBINED_PATH}/{CELL_TYPE}').mkdir(parents=True, exist_ok=True)
comb_df.to_excel(
    f'{COMBINED_PATH}/{CELL_TYPE}/{CELL_NAME}.xlsx',
    index=False
)

In [ ]:
y_real = comb_df['spike_type'].map({
    'AIS': 0,
    'UND': 1,
    'RA': 2,
}).values
y_hard = comb_df['hard_label'].map({
    'AIS': 0,
    'unknown': 1,
    'conflict': 1,
    'RA': 2,
}).values
y_soft = comb_df['soft_label'].map({
    'AIS': 0,
    'unknown': 1,
    'conflict': 1,
    'RA': 2,
}).values

In [ ]:
cm_hard = np.zeros((3, 3), dtype=int)
cm_soft = np.zeros((3, 3), dtype=int)
for r, h, s in zip(y_real, y_hard, y_soft):
    cm_hard[h, r] += 1
    cm_soft[s, r] += 1
print("Confusion Matrix - Hard Predictions:")
for row in cm_hard:
    print(' '.join(f'{x:5d}' for x in row))
print("Confusion Matrix - Soft Predictions:")
for row in cm_soft:
    print(' '.join(f'{x:5d}' for x in row))

In [ ]:
acc_hard = np.sum(y_real == y_hard) / len(y_real)
print(f'Accuracy - Hard Predictions: {acc_hard:.4f}')

acc_soft = np.sum(y_real == y_soft) / len(y_real)
print(f'Accuracy - Soft Predictions: {acc_soft:.4f}')

In [ ]:
print("Hard prediction")
print(f"AIS recall: {cm_hard[0, 0] / np.sum(cm_hard[:, 0]):.4f}")
print(f"AIS precision: {cm_hard[0, 0] / np.sum(cm_hard[0, :]):.4f}")
print(f"RA recall: {cm_hard[2, 2] / np.sum(cm_hard[:, 2]):.4f}")
print(f"RA precision: {cm_hard[2, 2] / np.sum(cm_hard[2, :]):.4f}")

In [ ]:
print("Soft prediction")
print(f"AIS recall: {cm_soft[0, 0] / np.sum(cm_soft[:, 0]):.4f}")
print(f"AIS precision: {cm_soft[0, 0] / np.sum(cm_soft[0, :]):.4f}")
print(f"RA recall: {cm_soft[2, 2] / np.sum(cm_soft[:, 2]):.4f}")
print(f"RA precision: {cm_soft[2, 2] / np.sum(cm_soft[2, :]):.4f}")